# 8장. MCP(Model Context Protocol) Tutorial

이 문서는 LLM 애플리케이션과 외부 시스템 간의 상호작용을 위한 표준 프로토콜인 **모델 컨텍스트 프로토콜(MCP)**의 파이썬 SDK(`FastMCP`) 사용법을 다루는 종합 튜토리얼이다.

이 노트북을 통해 다음을 학습한다:
- MCP의 핵심 개념: **도구, 리소스, 프롬프트**
- `FastMCP`를 사용한 간단한 MCP 서버 구축
- 프로세스 내부 클라이언트를 이용한 서버 기능 테스트
- 외부 API를 연동하는 실용적인 예제 작성

## 1: 환경 설정

먼저 MCP 파이썬 SDK를 설치한다. `[cli]` 옵션은 서버 실행을 위한 명령줄 도구를 함께 설치한다.

In [2]:
%pip install "mcp[cli]" -q

Note: you may need to restart the kernel to use updated packages.


## 2: 세 가지 핵심 구성 요소

모든 MCP 서버는 **도구, 리소스, 프롬프트**라는 세 가지 핵심 요소를 통해 LLM에 기능을 제공한다.

### 도구 (Tools): LLM의 '행동'

LLM이 실행할 수 있는 함수다. 계산, 데이터베이스 업데이트, 외부 API 호출처럼 시스템의 상태를 변경하는 작업을 담당한다.

```python
# 예시 코드 (실행하지 않음)
@mcp.tool()
def add(a: int, b: int) -> int:
     """두 개의 정수를 더하여 그 결과를 반환한다."""
     return a + b
```

### 리소스 (Resources): LLM의 '지식'

LLM이 읽을 수 있는 읽기 전용 데이터 엔드포인트다. 정적 설정 파일, 사용자 프로필 등 부수 효과가 없는 컨텍스트를 제공한다.

```python
# 예시 코드 (실행하지 않음)
@mcp.resource("data://config")
def get_app_config() -> dict:
     """애플리케이션의 현재 설정 정보를 반환한다."""
     return {"version": "1.0"}
```

### 프롬프트 (Prompts): 재사용 가능한 '대화'

특정 작업을 위한 대화 구조나 지침을 미리 정의해 둔 템플릿이다. 이를 통해 LLM이 일관된 방식으로 특정 과업을 수행하도록 유도할 수 있다.

```python
# 예시 코드 (실행하지 않음)
@mcp.prompt("summarize_text")
async def summarize_prompt(text: str) -> list[dict]:
     """텍스트 요약 프롬프트를 생성한다."""
     return [
         {"role": "system", "content": "당신은 텍스트 요약 전문가다."},
         {"role": "user", "content": f"다음 텍스트를 요약해줘: {text}"}
     ]
```

## 3: 단계별 MCP 서버 구축 및 테스트

이제 실제로 동작하는 MCP 서버를 만들고, 프로세스 내부 클라이언트를 이용해 테스트한다. 아래 코드는 서버 정의와 테스트 코드를 모두 포함하고 있다.

#### server.py

In [ ]:
from fastmcp import FastMCP

# 1. 서버 인스턴스화
mcp = FastMCP(
    name="나의 첫 MCP 서버",
    instructions="이 서버는 기본적인 계산 도구와 사용자 정보 리소스를 제공한다."
)

# 2. 도구 추가
@mcp.tool()
def add(a: int, b: int) -> int:
    """두 개의 정수를 더하여 그 결과를 반환한다."""
    print(f"[Server] 'add' 도구 호출됨: a={a}, b={b}")
    return a + b

# 3. 리소스 추가
USER_DATA = { "101": {"name": "Alice", "email": "alice@example.com"} }

# 정적 리소스
@mcp.resource("data://config")
def get_app_config() -> dict:
    """애플리케이션의 현재 설정 정보를 반환한다."""
    print("[Server] 'data://config' 리소스 요청됨")
    return {"theme": "dark", "version": "1.2"}

# 동적 리소스
@mcp.resource("users://{user_id}/profile")
async def get_user_profile(user_id: str) -> dict:
    """지정된 ID의 사용자 프로필을 반환한다."""
    print(f"[Server] 'users://{user_id}/profile' 리소스 요청됨")
    return USER_DATA.get(user_id, {})

# 4. 프롬프트 추가
@mcp.prompt("summarize_text")
async def summarize_prompt(text: str) -> list[dict]:
    """제공된 텍스트를 요약하는 프롬프트를 생성한다."""
    print("[Server] 'summarize_text' 프롬프트 요청됨")
    return [
        {"role": "system", "content": "당신은 텍스트를 간결하게 요약하는 데 능숙한 AI 어시스턴트다."},
        {"role": "user", "content": f"다음 텍스트를 세 문장으로 요약해 주세요:\n\n{text}"}
    ]

# 서버 직접 실행 시
if __name__ == "__main__":
    mcp.run()

#### test_server.py

In [ ]:
from fastmcp import Client
import asyncio
from server import mcp  # 서버 파일에서 mcp 객체 import

async def test_server_in_process():
    """MCP 서버의 모든 기능을 테스트하는 함수"""
    print("\n--- 💡 프로세스 내부 로컬 테스트 시작 ---")
    
    # 클라이언트가 메모리에 있는 mcp 서버 객체를 직접 가리키도록 설정한다.
    client = Client(mcp)
    
    async with client:
        # 'add' 도구 호출
        print("\n[Client] 'add' 도구 호출...")
        add_result = await client.call_tool("add", {"a": 15, "b": 27})
        print(f"[Client] 'add' 도구 결과: {add_result}")
        
        # 정적 리소스 읽기
        print("\n[Client] 'config' 리소스 읽기...")
        config_data = await client.read_resource("data://config")
        print(f"[Client] 'config' 리소스 결과: {config_data}")
        
        # 동적 리소스 읽기
        print("\n[Client] 'user 101' 리소스 읽기...")
        user_profile = await client.read_resource("users://101/profile")
        print(f"[Client] 'user 101' 리소스 결과: {user_profile}")
        
        # 프롬프트 구조 얻기
        print("\n[Client] 'summarize_text' 프롬프트 요청...")
        prompt_messages = await client.get_prompt("summarize_text", {"text": "MCP는 LLM과 외부 시스템 간의 표준화된 통신을 정의하는 프로토콜이다."})
        print(f"[Client] 'summarize_text' 프롬프트 결과: {prompt_messages}")
        
    print("\n--- ✅ 프로세스 내부 로컬 테스트 종료 ---")

async def test_individual_features():
    """개별 기능을 상세하게 테스트하는 함수"""
    print("\n--- 🔍 개별 기능 상세 테스트 시작 ---")
    
    client = Client(mcp)
    
    async with client:
        print("\n1. 도구 테스트:")
        print("=" * 30)
        
        # 여러 계산 테스트
        test_cases = [
            {"a": 5, "b": 3},
            {"a": 100, "b": 200},
            {"a": -10, "b": 25},
            {"a": 0, "b": 0}
        ]
        
        for case in test_cases:
            result = await client.call_tool("add", case)
            print(f"add({case['a']}, {case['b']}) = {result}")
        
        print("\n2. 리소스 테스트:")
        print("=" * 30)
        
        # 정적 리소스 테스트
        config = await client.read_resource("data://config")
        print(f"설정 정보: {config}")
        
        # 동적 리소스 테스트 (존재하는 사용자)
        user_101 = await client.read_resource("users://101/profile")
        print(f"사용자 101: {user_101}")
        
        # 동적 리소스 테스트 (존재하지 않는 사용자)
        user_999 = await client.read_resource("users://999/profile")
        print(f"사용자 999: {user_999}")
        
        print("\n3. 프롬프트 테스트:")
        print("=" * 30)
        
        # 다양한 텍스트로 프롬프트 테스트
        test_texts = [
            "FastMCP는 Python으로 MCP 서버를 쉽게 구축할 수 있는 프레임워크다.",
            "인공지능은 인간의 지능을 모방하여 학습하고 추론하는 컴퓨터 시스템이다.",
            "짧은 텍스트"
        ]
        
        for i, text in enumerate(test_texts, 1):
            prompt = await client.get_prompt("summarize_text", {"text": text})
            print(f"프롬프트 {i}: {len(prompt)} 메시지")
            print(f"  시스템: {prompt[0]['content'][:50]}...")
            print(f"  사용자: {prompt[1]['content'][:50]}...")
    
    print("\n--- ✅ 개별 기능 상세 테스트 종료 ---")

def main():
    """메인 테스트 실행 함수"""
    print("🚀 MCP 서버 테스트 시작")
    
    # 기본 테스트 실행
    asyncio.run(test_server_in_process())
    
    # 상세 테스트 실행
    asyncio.run(test_individual_features())
    
    print("\n🎉 모든 테스트 완료!")

if __name__ == "__main__":
    main()

## 4: 종합 실습 - 날씨 정보 도구 만들기

지금까지 배운 내용을 통합하여 외부 API를 호출하는 실용적인 날씨 정보 도구를 만들어본다. 이 예제는 `httpx` 라이브러리를 사용하여 무료 날씨 API(`wttr.in`)에서 데이터를 가져온다.

In [5]:
# 실습에 필요한 httpx 라이브러리를 설치한다.
%pip install httpx -q

Note: you may need to restart the kernel to use updated packages.


In [5]:
from mcp.server.fastmcp import FastMCP
import asyncio

# 1. 서버 인스턴스화
mcp = FastMCP(
    name="나의 첫 MCP 서버"
)

# 2. 도구 추가
@mcp.tool()
def add(a: int, b: int) -> int:
    """두 개의 정수를 더하여 그 결과를 반환한다."""
    print(f"[Server] 'add' 도구 호출됨: a={a}, b={b}")
    return a + b

# 3. 리소스 추가
USER_DATA = { "101": {"name": "Alice", "email": "alice@example.com"} }

# 정적 리소스
@mcp.resource("data://config")
def get_app_config() -> dict:
    """애플리케이션의 현재 설정 정보를 반환한다."""
    print("[Server] 'data://config' 리소스 요청됨")
    return {"theme": "dark", "version": "1.2"}

# 동적 리소스
@mcp.resource("users://{user_id}/profile")
async def get_user_profile(user_id: str) -> dict:
    """지정된 ID의 사용자 프로필을 반환한다."""
    print(f"[Server] 'users://{user_id}/profile' 리소스 요청됨")
    return USER_DATA.get(user_id, {})

# 4. 프롬프트 추가
@mcp.prompt("summarize_text")
async def summarize_prompt(text: str) -> list[dict]:
    """제공된 텍스트를 요약하는 프롬프트를 생성한다."""
    print("[Server] 'summarize_text' 프롬프트 요청됨")
    return [
        {"role": "system", "content": "당신은 텍스트를 간결하게 요약하는 데 능숙한 AI 어시스턴트다."},
        {"role": "user", "content": f"다음 텍스트를 세 문장으로 요약해 주세요:\n\n{text}"}
    ]

# 서버 실행
#if __name__ == "__main__":
#    mcp.run()

## 참고 사항

이 튜토리얼을 통해 MCP의 기본 개념과 `FastMCP`를 사용한 서버 구축 방법을 익혔다. 더 깊이 학습하려면 공식 문서를 참고하는 것을 추천한다.

- **MCP 공식 GitHub**: [https://github.com/modelcontextprotocol/python-sdk](https://github.com/modelcontextprotocol/python-sdk)